## Setup

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
raw_file_path = Path(
    "Datasets",
    "ListofNEALicensedEatingEstablishments.csv"
)

processed_folder_path = Path("Processed Data")

processed_file_path = processed_folder_path / "nea_licensed_eating_establishments_clean.csv"

print("Raw file:")
print(raw_file_path)

print("\nProcessed file:")
print(processed_file_path)

Raw file:
Datasets\ListofNEALicensedEatingEstablishments.csv

Processed file:
Processed Data\nea_licensed_eating_establishments_clean.csv


In [3]:
if raw_file_path.exists():
    print("NEA dataset found.")
else:
    print("NEA dataset not found.")

NEA dataset found.


In [4]:
nea_raw = pd.read_csv(raw_file_path)

nea_raw.head()

,licensee_name,licence_number,premises_address,grade,demerit_points,suspension_start_date,suspension_end_date
0,REPUBLIC HOTELS & RESORTS LIMITED,W99288X000,392 HAVELOCK ROAD (FISH/MEAT) GRAND COPTHORNE ...,A,na,na,na
1,REPUBLIC HOTELS & RESORTS LIMITED,W99281K000,392 HAVELOCK ROAD (BANQUET KITCHEN) GRAND COPT...,A,na,na,na
2,M.K. RAMA PTE LTD,W96344L000,37 NORRIS ROAD SINGAPORE 208279,A,na,na,na
3,GRAND PARK PROPERTY PTE. LTD.,W96230L000,10 COLEMAN STREET (2ND STOREY) GRAND PARK CITY...,A,na,na,na
4,MILLENIA PRIVATE LIMITED,W96214L000,2 TEMASEK BOULEVARD #B1-00 CONRAD CENTENNIAL S...,A,na,na,na


## Initial Checks and Data Exploration

In [5]:
print("Number of rows and columns:")
print(nea_raw.shape)

print("\nColumn names:")
print(nea_raw.columns.tolist())

print("\nData types:")
print(nea_raw.dtypes)

Number of rows and columns:
(36687, 7)

Column names:
['licensee_name', 'licence_number', 'premises_address', 'grade', 'demerit_points', 'suspension_start_date', 'suspension_end_date']

Data types:
licensee_name            object
licence_number           object
premises_address         object
grade                    object
demerit_points           object
suspension_start_date    object
suspension_end_date      object
dtype: object


In [6]:
print("Grade values:")
print(nea_raw["grade"].unique())

print("\n'na' counts per column:")
print((nea_raw == "na").sum())

print("\nNumber of exact duplicate rows:")
print(nea_raw.duplicated().sum())

Grade values:
['A' 'B' 'na' 'C']

'na' counts per column:
licensee_name                0
licence_number               0
premises_address             0
grade                     5302
demerit_points           35083
suspension_start_date    36567
suspension_end_date      36567
dtype: int64

Number of exact duplicate rows:
5


In [7]:
nea = nea_raw.copy()

nea.head()

,licensee_name,licence_number,premises_address,grade,demerit_points,suspension_start_date,suspension_end_date
0,REPUBLIC HOTELS & RESORTS LIMITED,W99288X000,392 HAVELOCK ROAD (FISH/MEAT) GRAND COPTHORNE ...,A,na,na,na
1,REPUBLIC HOTELS & RESORTS LIMITED,W99281K000,392 HAVELOCK ROAD (BANQUET KITCHEN) GRAND COPT...,A,na,na,na
2,M.K. RAMA PTE LTD,W96344L000,37 NORRIS ROAD SINGAPORE 208279,A,na,na,na
3,GRAND PARK PROPERTY PTE. LTD.,W96230L000,10 COLEMAN STREET (2ND STOREY) GRAND PARK CITY...,A,na,na,na
4,MILLENIA PRIVATE LIMITED,W96214L000,2 TEMASEK BOULEVARD #B1-00 CONRAD CENTENNIAL S...,A,na,na,na


## Cleaning Missing Values

In [8]:
na_columns = [
    "grade",
    "demerit_points",
    "suspension_start_date",
    "suspension_end_date"
]

for column in na_columns:
    nea[column] = nea[column].replace("na", pd.NA)

In [9]:
nea["demerit_points"] = pd.to_numeric(
    nea["demerit_points"],
    errors="coerce"
)

In [10]:
print("Missing values after cleaning:")
print(nea[na_columns].isnull().sum())

print("\ndemerit_points data type:")
print(nea["demerit_points"].dtype)

Missing values after cleaning:
grade                     5302
demerit_points           35083
suspension_start_date    36567
suspension_end_date      36567
dtype: int64

demerit_points data type:
float64


## Removing Duplicate Rows

In [11]:
print("Exact duplicate rows:")
print(nea.duplicated().sum())

Exact duplicate rows:
5


In [12]:
nea = nea.drop_duplicates()

nea = nea.reset_index(drop=True)

print("Shape after dropping duplicates:")
print(nea.shape)

Shape after dropping duplicates:
(36682, 7)


## Deriving Postal District

In [13]:
# Singapore postal sectors grouped by postal district
district_sectors = {
    "D01": ["01", "02", "03", "04", "05", "06"],
    "D02": ["07", "08"],
    "D03": ["14", "15", "16"],
    "D04": ["09", "10"],
    "D05": ["11", "12", "13"],
    "D06": ["17"],
    "D07": ["18", "19"],
    "D08": ["20", "21"],
    "D09": ["22", "23"],
    "D10": ["24", "25", "26", "27"],
    "D11": ["28", "29", "30"],
    "D12": ["31", "32", "33"],
    "D13": ["34", "35", "36", "37"],
    "D14": ["38", "39", "40", "41"],
    "D15": ["42", "43", "44", "45"],
    "D16": ["46", "47", "48"],
    "D17": ["49", "50", "81"],
    "D18": ["51", "52"],
    "D19": ["53", "54", "55", "82"],
    "D20": ["56", "57"],
    "D21": ["58", "59"],
    "D22": ["60", "61", "62", "63", "64"],
    "D23": ["65", "66", "67", "68"],
    "D24": ["69", "70", "71"],
    "D25": ["72", "73"],
    "D26": ["77", "78"],
    "D27": ["75", "76"],
    "D28": ["79", "80"]
}

sector_to_district = {}
for district, sectors in district_sectors.items():
    for sector in sectors:
        sector_to_district[sector] = district

In [14]:
# Some addresses have extra text after the postal code (e.g. stray brackets/quotes),
# so we take the last 6-digit number found in the address rather than anchoring to the end
postal_code_matches = nea["premises_address"].str.findall(r"\d{6}")

nea["postal_code"] = postal_code_matches.apply(
    lambda matches: matches[-1] if matches else None
)

nea["postal_sector"] = nea["postal_code"].str[:2]

nea["postal_district"] = nea["postal_sector"].map(sector_to_district)

In [15]:
unmatched_postal_district = nea[nea["postal_district"].isnull()]

print("Rows with no postal code found in address:")
print(len(unmatched_postal_district))

unmatched_postal_district[["premises_address"]].head()

Rows with no postal code found in address:
6083


,premises_address
630,CHOA CHU KANG PARK CHOA CHU KANG DRIVE
1118,101 MOUNT FABER PARK KIOSK FABER POINT
1626,391 ORCHARD ROAD #04-20A/21TAKASHIMAYA SHOPPIN...
1913,2 STAMFORD ROADLEVEL 68 & 69EQUINOX RESTAURANT
1941,6 RAFFLES BLVD 4TH STOREY MARINA MANDARIN


In [16]:
nea = nea.drop(columns=[
    "postal_code",
    "postal_sector"
])

nea.head()

,licensee_name,licence_number,premises_address,grade,demerit_points,suspension_start_date,suspension_end_date,postal_district
0,REPUBLIC HOTELS & RESORTS LIMITED,W99288X000,392 HAVELOCK ROAD (FISH/MEAT) GRAND COPTHORNE ...,A,NaN,<NA>,<NA>,D03
1,REPUBLIC HOTELS & RESORTS LIMITED,W99281K000,392 HAVELOCK ROAD (BANQUET KITCHEN) GRAND COPT...,A,NaN,<NA>,<NA>,D03
2,M.K. RAMA PTE LTD,W96344L000,37 NORRIS ROAD SINGAPORE 208279,A,NaN,<NA>,<NA>,D08
3,GRAND PARK PROPERTY PTE. LTD.,W96230L000,10 COLEMAN STREET (2ND STOREY) GRAND PARK CITY...,A,NaN,<NA>,<NA>,D06
4,MILLENIA PRIVATE LIMITED,W96214L000,2 TEMASEK BOULEVARD #B1-00 CONRAD CENTENNIAL S...,A,NaN,<NA>,<NA>,D01


## Final Cleaned Dataset

In [17]:
print("Cleaned dataset shape:")
print(nea.shape)

print("\nCleaned data types:")
print(nea.dtypes)

print("\nMissing values:")
print(nea.isnull().sum())

print("\nExact duplicate rows:")
print(nea.duplicated().sum())

Cleaned dataset shape:
(36682, 8)

Cleaned data types:
licensee_name             object
licence_number            object
premises_address          object
grade                     object
demerit_points           float64
suspension_start_date     object
suspension_end_date       object
postal_district           object
dtype: object

Missing values:
licensee_name                0
licence_number               0
premises_address             0
grade                     5302
demerit_points           35079
suspension_start_date    36562
suspension_end_date      36562
postal_district           6083
dtype: int64

Exact duplicate rows:
0


In [18]:
display(nea.head(10))

,licensee_name,licence_number,premises_address,grade,demerit_points,suspension_start_date,suspension_end_date,postal_district
0,REPUBLIC HOTELS & RESORTS LIMITED,W99288X000,392 HAVELOCK ROAD (FISH/MEAT) GRAND COPTHORNE ...,A,NaN,<NA>,<NA>,D03
1,REPUBLIC HOTELS & RESORTS LIMITED,W99281K000,392 HAVELOCK ROAD (BANQUET KITCHEN) GRAND COPT...,A,NaN,<NA>,<NA>,D03
2,M.K. RAMA PTE LTD,W96344L000,37 NORRIS ROAD SINGAPORE 208279,A,NaN,<NA>,<NA>,D08
3,GRAND PARK PROPERTY PTE. LTD.,W96230L000,10 COLEMAN STREET (2ND STOREY) GRAND PARK CITY...,A,NaN,<NA>,<NA>,D06
4,MILLENIA PRIVATE LIMITED,W96214L000,2 TEMASEK BOULEVARD #B1-00 CONRAD CENTENNIAL S...,A,NaN,<NA>,<NA>,D01
5,MILLENIA PRIVATE LIMITED,W96212P000,2 TEMASEK BOULEVARD (MAIN KITCHEN) #02-00 CONR...,A,NaN,<NA>,<NA>,D01
6,BCH HOTEL INVESTMENT PTE LTD,W95267N000,80 MIDDLE ROAD HOTEL INTER-CONTINENTAL BASEMEN...,A,NaN,<NA>,<NA>,D07
7,RCMS PROPERTIES PTE LTD,W95169X000,7 RAFFLES AVENUE (BANQUET FINISHING KITCHEN) R...,A,NaN,<NA>,<NA>,D01
8,RCMS PROPERTIES PTE LTD,W95163J000,7 RAFFLES AVENUE (PRODUCTION KITCHEN) RITZ CAR...,A,NaN,<NA>,<NA>,D01
9,SUNTEC SINGAPORE INTERNATIONAL CONVENTION & EX...,W95017C000,1 RAFFLES BOULEVARD #06-00 SINGAPORE INT'L CON...,A,NaN,<NA>,<NA>,D01


In [19]:
display(nea.tail(10))

,licensee_name,licence_number,premises_address,grade,demerit_points,suspension_start_date,suspension_end_date,postal_district
36672,SYED ABDUL RAHAMAN S/O M MOHD AHDAM,ADF0110001,ADAM ROAD FOOD CENTRE Stall No 01-10,A,NaN,<NA>,<NA>,NaN
36673,SUMADI BIN SAPARI,ADF0109002,ADAM ROAD FOOD CENTRE Stall No 01-09,B,NaN,<NA>,<NA>,NaN
36674,IRDAWATY BINTE MOHD ALI,ADF0108002,ADAM ROAD FOOD CENTRE Stall No 01-08,B,NaN,<NA>,<NA>,NaN
36675,ZAITON BINTE AHMAD,ADF0107002,ADAM ROAD FOOD CENTRE Stall No 01-07,B,NaN,<NA>,<NA>,NaN
36676,MOHD HANAFIAH BIN MOHD IDRIS,ADF0106001,ADAM ROAD FOOD CENTRE Stall No 01-06,A,NaN,<NA>,<NA>,NaN
36677,AINON BTE BADRI ( AINON BTE ALI ),ADF0105001,ADAM ROAD FOOD CENTRE Stall No 01-05,B,NaN,<NA>,<NA>,NaN
36678,SYED IBRAHIM BIN PEER MOHAMED,ADF0104001,ADAM ROAD FOOD CENTRE Stall No 01-04,B,NaN,<NA>,<NA>,NaN
36679,SAITON BINTE ALI,ADF0103001,ADAM ROAD FOOD CENTRE Stall No 01-03,B,NaN,<NA>,<NA>,NaN
36680,AMINAH BTE K OMAR,ADF0102001,ADAM ROAD FOOD CENTRE Stall No 01-02,B,NaN,<NA>,<NA>,NaN
36681,AISHA BEGAM BINTE MOHAMED MUSTHAPA,ADF0101002,ADAM ROAD FOOD CENTRE Stall No 01-01,B,NaN,<NA>,<NA>,NaN


In [20]:
processed_folder_path.mkdir(
    parents=True,
    exist_ok=True
)

print("Processed Data folder is ready.")

Processed Data folder is ready.


In [21]:
nea.to_csv(
    processed_file_path,
    index=False
)

print("Cleaned NEA dataset saved successfully.")
print(processed_file_path)

Cleaned NEA dataset saved successfully.
Processed Data\nea_licensed_eating_establishments_clean.csv


In [22]:
if processed_file_path.exists():
    print("Saved file found.")
else:
    print("Saved file was not created.")

Saved file found.


## Verify Saved File

In [23]:
nea_saved_check = pd.read_csv(
    processed_file_path
)

print("Saved dataset shape:")
print(nea_saved_check.shape)

display(nea_saved_check.head())

Saved dataset shape:
(36682, 8)


,licensee_name,licence_number,premises_address,grade,demerit_points,suspension_start_date,suspension_end_date,postal_district
0,REPUBLIC HOTELS & RESORTS LIMITED,W99288X000,392 HAVELOCK ROAD (FISH/MEAT) GRAND COPTHORNE ...,A,NaN,NaN,NaN,D03
1,REPUBLIC HOTELS & RESORTS LIMITED,W99281K000,392 HAVELOCK ROAD (BANQUET KITCHEN) GRAND COPT...,A,NaN,NaN,NaN,D03
2,M.K. RAMA PTE LTD,W96344L000,37 NORRIS ROAD SINGAPORE 208279,A,NaN,NaN,NaN,D08
3,GRAND PARK PROPERTY PTE. LTD.,W96230L000,10 COLEMAN STREET (2ND STOREY) GRAND PARK CITY...,A,NaN,NaN,NaN,D06
4,MILLENIA PRIVATE LIMITED,W96214L000,2 TEMASEK BOULEVARD #B1-00 CONRAD CENTENNIAL S...,A,NaN,NaN,NaN,D01
